# Gold Layer - Hospital Analytics

## Read Silver Delta Tables

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

patient_silver = spark.read.format("delta").load(
    "/Volumes/workspace/default/final_assignment/silver/patients"
)

visit_silver = spark.read.format("delta").load(
    "/Volumes/workspace/default/final_assignment/silver/visits"
)


## Patient Summary

In [0]:
patient_summary = (
    visit_silver
    .groupBy("patient_id")
    .agg(
        count("visit_id").alias("total_visits"),
        sum("bill_amount").alias("total_bill")
    )
)

## Validate Patient Summary

In [0]:
display(
    patient_summary.limit(15)
)

print(
    "Patient Summary Count",
    patient_summary.count()
)

patient_id,total_visits,total_bill
PAT000067,3,5471.3
PAT000086,1,1246.72
PAT000286,1,2747.6
PAT000683,2,3020.46
PAT002182,1,800.65
PAT003066,1,1258.22
PAT003246,1,848.12
PAT003384,2,3937.13
PAT003610,1,2150.6
PAT004081,2,1662.6399999999999


Patient Summary Count 60000


## Doctor Performance

In [0]:
doctor_performance = (
    visit_silver
    .groupBy("doctor_id")
    .agg(
        countDistinct("patient_id").alias("patients_handled"),
        sum("bill_amount").alias("revenue_generated")
    )
)

## Validate Doctor Performance

In [0]:
display(
    doctor_performance.limit(15)
)

print(
    "Doctor Performance Count",
    doctor_performance.count()
)

doctor_id,patients_handled,revenue_generated
PRO00918,136,189532.04
PRO01126,43,60371.030000000006
PRO00591,35,54150.14000000001
PRO01322,32,42568.24
PRO01280,25,28087.790000000005
PRO00070,223,313455.74
PRO01116,42,75365.26000000001
PRO00045,215,295421.51
PRO00363,32,57628.40999999999
PRO00321,36,46485.61


Doctor Performance Count 1491


## Rank Top Doctors by Revenue

In [0]:
rank_window = Window.orderBy(
    col("revenue_generated").desc()
)

doctor_performance = (
    doctor_performance
    .withColumn(
        "doctor_rank",
        dense_rank().over(rank_window)
    )
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


## Validate Doctor Ranking

In [0]:
display(
    doctor_performance
    .orderBy("doctor_rank")
    .limit(15)
)

print(
    "Doctor Performance Ranked Count",
    doctor_performance.count()
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


doctor_id,patients_handled,revenue_generated,doctor_rank
PRO00033,255,401810.44000000006,1
PRO00031,263,390238.3299999997,2
PRO00009,248,387909.45,3
PRO00004,263,386121.4499999998,4
PRO00032,251,385783.2700000001,5
PRO00006,256,380368.67999999976,6
PRO00069,239,375174.98,7
PRO00007,259,372621.14000000013,8
PRO00062,244,370986.77000000014,9
PRO00021,225,364141.8599999999,10


Doctor Performance Ranked Count 1491


## Write Gold Tables

### Patient Summary

In [0]:
patient_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .save("/Volumes/workspace/default/final_assignment/gold/patient_summary")

### Doctor Performance

In [0]:
doctor_performance.write \
    .format("delta") \
    .mode("overwrite") \
    .save("/Volumes/workspace/default/final_assignment/gold/doctor_performance")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


## Verify Gold Layer

In [0]:
display(
    spark.read.format("delta")
    .load("/Volumes/workspace/default/final_assignment/gold/patient_summary").limit(15)
)

patient_id,total_visits,total_bill
PAT000067,3,5471.3
PAT000086,1,1246.72
PAT000286,1,2747.6
PAT000683,2,3020.46
PAT002182,1,800.65
PAT003066,1,1258.22
PAT003246,1,848.12
PAT003384,2,3937.13
PAT003610,1,2150.6
PAT004081,2,1662.6399999999999


In [0]:
display(
    spark.read.format("delta")
    .load("/Volumes/workspace/default/final_assignment/gold/doctor_performance").limit(15)
)

doctor_id,patients_handled,revenue_generated,doctor_rank
PRO00033,255,401810.44000000006,1
PRO00031,263,390238.3299999997,2
PRO00009,248,387909.45,3
PRO00004,263,386121.4499999998,4
PRO00032,251,385783.2700000001,5
PRO00006,256,380368.67999999976,6
PRO00069,239,375174.98,7
PRO00007,259,372621.14000000013,8
PRO00062,244,370986.77000000014,9
PRO00021,225,364141.8599999999,10
